# SKKU 2024 Challenge
## Quantum State Tomography

In [1]:
import numpy as np
import qiskit
from qiskit_aer import Aer

In [25]:
## utils.py
backend = Aer.get_backend("aer_simulator")
def create_1q_target_circuit(qc_data):
    qc = qiskit.QuantumCircuit(1)
    qc.ry(qc_data[0],0)
    qc.rz(qc_data[1],0)
    return qc

def create_2q_target_circuit(qc_data):
    qc = qiskit.QuantumCircuit(2)
    for i in range(2):
        qc.ry(qc_data[2*i+1],i)
        qc.rz(qc_data[2*i+1],i)
    qc.cz(0,1)
    qc.rz(qc_data[4],1)
    qc.cx(0,1)
    for i in range(2):
        qc.ry(qc_data[2*i+5],i)
        qc.rz(qc_data[2*i+5],i)
    
    return qc

backend2 = Aer.get_backend("statevector_simulator")
def give_score(U_hidden, U_gen):
    qc = U_hidden.compose(U_gen.inverse())
    result = backend2.run(qc).result()
    score = result.get_statevector()[0]
    return np.abs(score)

## TODO : make error handler

def generate_clifford_circ(num_qubits, depth):
    U_hidden = qiskit.QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        U_hidden.h(i)
    count = 0
    s1s2 = {-1,-1}
    while(count < depth):
        site_i = int(np.random.rand()*num_qubits)
        site_j = int(np.random.rand()*num_qubits)
        if(site_i == site_j or {site_i, site_j} == s1s2):
            continue
        print(site_i, site_j, count)
        count += 1
        U_hidden.cz(site_i, site_j)
    return U_hidden

### Problem 1

In [3]:
random_vec = np.random.rand(2)*2*np.pi
#random_vec = np.array([np.pi/2,0])
U_hidden = create_1q_target_circuit(random_vec)
print(random_vec)

[4.54375896 6.10494112]


In [ ]:
U_hidden.draw()

┌────────────┐┌────────────┐
q: ┤ Ry(4.5438) ├┤ Rz(6.1049) ├
   └────────────┘└────────────┘

#### Part I

##### a)

In a given single qubit pure state can be written as:
$$
\ket{\psi} = a\ket{0} + b e^{i\theta} \ket{1}
$$
where $a, b, \theta$ are real parameter subject to $a^2 + b^2 =1$ and $0 \leq \theta < 2\pi$. 
In order to determine these three parameter, it requires at least three measurement. 

Our measurement is performed via POVM. There are detectors whose number is equal to the number of qubits. 
In this scenerio, we perform $z$-basis projection measurement. Therefore, the POVM becomes $\left\{ \ket{0}\bra{0}, \ket{1}\bra{1}\right\}$, each corresponds to the presense of signal or not. 
Let $\hat{m}_0, \hat{m}_1$ denote the estimated probability of occuring signals. 
We can add another operation $U$ before the measurement and after the computational stage(here, state preparation), we can perform measurement another POVM basis, $\left\{ U^{\dagger} \ket{0}\bra{0} U, U^{\dagger} \ket{1}\bra{1} U\right\}$.

In first measurement, to determine $a, b$, we can directly perform measurement without any unitary operation. That is $U^1 = I$, here. 
Then, we see:
$$
\begin{align}
	\hat{m}_0 &\simeq  \left| \bra{\psi}\ket{0} \right|^2 &= a^2 \\
	\hat{m}_1 &\simeq  \left| \bra{\psi}\ket{1} \right|^2 &= b^2 \\
\end{align}
$$
Therefore, we can estimate,
$$
\begin{align}
	a &= \sqrt{\hat{m}_0} \\
	b &= \sqrt{\hat{m}_1} \\
\end{align}
$$
On the other hand, we need two additional measurement. 
Here let $U_2 = H$, where $H$ is a hadamard gate.
$$
H = \frac{1}{\sqrt{2}}
\begin{pmatrix}
	1 & 1 \\
	1 & -1
\end{pmatrix}.
$$
Then, our POVM becomes $\left\{ \ket{+}\bra{+}, \ket{-}\bra{-}\right\}$ where $\ket{+}, \ket{-}$ are the eigenstate of Pauli-X matrix whose eigenvalue is $1,-1$ for each.
Here, let $\hat{m}_{+}, \hat{m}_{-}$ be the estimated probability of occuring signals.
$$
\begin{align}
	\hat{m}_{+} &\simeq  \left| \bra{\psi}\ket{+} \right|^2 &= \left| \frac{a + b e^{i\theta}}{2} \right|^2 &= \frac{1 + 2ab\cos\theta}{2}  \\
	\hat{m}_{-} &\simeq  \left| \bra{\psi}\ket{-} \right|^2 &= \left| \frac{a - b e^{i\theta}}{2} \right|^2 &= \frac{1 - 2ab\cos\theta}{2} 
\end{align}
$$
Or,
$$
\cos\theta = \frac{\hat{m}_{+} - \hat{m}_{-}}{2ab}
$$
Here, it is not enough yet. As there is a degree of freedom choosing a sign of $\sin\theta$. 
Finally, let $U_3 = S H$ where $S$ is S gate.
$$
S = \begin{pmatrix}
	1 & 0 \\
	0 & i 
\end{pmatrix}.
$$
Then our POVM becomes $\left\{ \ket{i}\bra{i}, \ket{-i}\bra{-i}\right\}$ where $\ket{i}, \ket{-i}$ are the eigenstate of Pauli-Y matrix whose eigenvalue is $1,-1$ for each.
Here, let $\hat{m}_{i}, \hat{m}_{-i}$ be the estimated probability of occuring signals.
$$
\begin{align}
	\hat{m}_{i} &\simeq  \left| \bra{\psi}\ket{i} \right|^2 &= \left| \frac{a + b e^{i(\theta + \frac{\pi}{2})}}{2} \right|^2 &= \frac{1 + 2ab\sin\theta}{2}  \\
	\hat{m}_{-i} &\simeq  \left| \bra{\psi}\ket{-i} \right|^2 &= \left| \frac{a - b e^{i\theta  + \frac{\pi}{2}}}{2} \right|^2 &= \frac{1 - 2ab\sin\theta}{2}  
\end{align}
$$
Or,
$$
\sin\theta = \frac{\hat{m}_{i} - \hat{m}_{-i}}{2ab}
$$

##### b)

Now, Let me explain how we reconstruct circuit parameter. 
In glance of action of $RY$ and $RZ$ gate, we see:
$$
RY(\phi_{y}) = 
\begin{pmatrix}
	\cos \frac{\phi_{y}}{2} & -\sin \frac{\phi_{y}}{2} \\
	\sin \frac{\phi_{y}}{2} & \cos \frac{\phi_{y}}{2}
\end{pmatrix}
$$
and 
$$
RZ(\phi_{z}) = 
\begin{pmatrix}
	1 & 0 \\
	0 & e^{i\phi_{z}}
\end{pmatrix}
$$
Therefore, we have:
$$
RZ(\phi_{z})RY(\phi_{y}) = \cos \frac{\phi_{y}}{2} \ket{0} + e^{i\phi_{z}} \sin \frac{\phi_{y}}{2} \ket{1} .
$$
Comparing this equation to the first given $\ket{\psi}$, we find $\phi_{y} = 2\arccos a$ with $\phi_{y} = 2\arcsin b$ and $\phi_{z} = \theta$.

In [5]:
qc1 = U_hidden.copy()
qc2 = qiskit.QuantumCircuit(1)
# Add some circuits
circ = qc1.compose(qc2)
circ.measure_all()
result = backend.run(circ, shots=100).result()
print(result.data())

{'counts': {'0x1': 62, '0x0': 38}}


In [6]:
shot = result.data()["counts"]
m0, m1 = shot["0x0"], shot["0x1"]
m0 /= 100; m1 /= 100
a, b = np.sqrt(m0), np.sqrt(m1)
a, b

(np.float64(0.6164414002968976), np.float64(0.7874007874011811))

In [7]:
qc3 = qiskit.QuantumCircuit(1)
qc3.h(0)
circ = qc1.compose(qc3)
circ.measure_all()
result = backend.run(circ, shots=100).result()
print(result.data())

{'counts': {'0x0': 2, '0x1': 98}}


In [8]:
circ.draw()

┌────────────┐┌────────────┐┌───┐ ░ ┌─┐
     q: ┤ Ry(4.5438) ├┤ Rz(6.1049) ├┤ H ├─░─┤M├
        └────────────┘└────────────┘└───┘ ░ └╥┘
meas: 1/═════════════════════════════════════╩═
                                             0

In [9]:
shot = result.data()["counts"]
m0, m1 = shot["0x0"], shot["0x1"]
m0 /= 100; m1 /= 100
theta = np.arccos((m0 - m1)/2/a/b)
#theta = np.arcsin
theta,(m0 - m1)/2/a/b

(np.float64(2.9924765003156746), np.float64(-0.988902772116395))

In [10]:
reconstructed_vec = np.array([2*np.arccos(a), theta])
U_gen = create_1q_target_circuit(reconstructed_vec)
U_gen.draw()

┌────────────┐┌────────────┐
q: ┤ Ry(1.8132) ├┤ Rz(2.9925) ├
   └────────────┘└────────────┘

In [11]:
give_score(U_hidden, U_gen)

np.float64(0.9992188931692645)

#### Part II

##### a)

In a given single qubit pure state can be written as:
$$
\ket{\psi} = a\ket{00} + b \ket{01} + c \ket{10} + d\ket{11}
$$
where $a$ is a real posivive number $b, c$ and $d$ are complex numbers subject to $a^2 + b^2 + c^2 + d^2 =1$ without loss of generality.
If the parameters are subjected to real numbers, it seems sufficient to idenify all the parameters $a, b, c$ and $d$ with running two circuits. 
First, we run circuit without any additional operation, $U=I$. 
Then our POVM bases are $\left\{ \ket{00}\bra{00}, \ket{01}\bra{01}, \ket{10}\bra{10}, \ket{11}\bra{11} \right\}$, which directly reproduce $a^2 , b^2 , c^2$ and $d^2$.
That is,
$$
\begin{align}
	\hat{m}_{00} &\simeq \left| \bra{\psi}\ket{00} \right|^2 &= a^2 \\ 
	\hat{m}_{01} &\simeq \left| \bra{\psi}\ket{01} \right|^2 &= b^2 \\ 
	\hat{m}_{10} &\simeq \left| \bra{\psi}\ket{10} \right|^2 &= c^2 \\ 
	\hat{m}_{11} &\simeq \left| \bra{\psi}\ket{11} \right|^2 &= d^2  
\end{align}
$$
where $\hat{m}_{00}, \hat{m}_{01}, \hat{m}_{10}, \hat{m}_{11}$ are the estimated probability from the measurement outcome, $\left\{ \ket{00}\bra{00}, \ket{01}\bra{01}, \ket{10}\bra{10}, \ket{11}\bra{11} \right\}$.
Next, we can add Hadamard operation both qubits, then the resulting POVM basis are $\left\{ \ket{++}\bra{++}, \ket{+-}\bra{+-}, \ket{-+}\bra{-+}, \ket{--}\bra{--} \right\}$.
Then, 
$$
\begin{align}
	\hat{m}_{++} &\simeq \left| \bra{\psi}\ket{++} \right|^2 &= \frac{(a+b+c+d)^2}{4} \\ 
	\hat{m}_{+-} &\simeq \left| \bra{\psi}\ket{+-} \right|^2 &= \frac{(a-b+c-d)^2}{4} \\ 
	\hat{m}_{-+} &\simeq \left| \bra{\psi}\ket{-+} \right|^2 &= \frac{(a+b-c-d)^2}{4} \\ 
	\hat{m}_{--} &\simeq \left| \bra{\psi}\ket{++} \right|^2 &= \frac{(a-b-c+d)^2}{4}  
\end{align}
$$
This results in the following equation:
$$
\begin{align}
	a+b+c+d &= \pm 2\sqrt{\hat{m}_{++}} \\ 
	a-b+c-d &= \pm 2\sqrt{\hat{m}_{+-}} \\ 
	a+b-c-d &= \pm 2\sqrt{\hat{m}_{-+}} \\ 
	a-b-c+d &= \pm 2\sqrt{\hat{m}_{--}} 
\end{align}
$$

##### b)

In a given single qubit pure state can be written as:
$$
\ket{\psi} = a\ket{00} + b e^{i\theta_{b}} \ket{01} + c e^{i\theta_{c}} \ket{10} + d e^{i\theta_{b}} \ket{11}
$$
where $a, b, c$ and $d$ are real posivive numbers $\theta_b , \theta_c , \theta_d$ are real numbers subject to $0 \le \theta_{b} , \theta_{c} , \theta_{d} <2\pi> $ without loss of generality.
If the parameters are subjected to real numbers, it seems sufficient to idenify all the parameters $a, b, c$ and $d$ with running two circuits. 
First, we run circuit without any additional operation, $U=I$. 
Then our POVM bases are $\left\{ \ket{00}\bra{00}, \ket{01}\bra{01}, \ket{10}\bra{10}, \ket{11}\bra{11} \right\}$, which directly reproduce $a^2 , b^2 , c^2$ and $d^2$.
That is,
$$
\begin{align}
	\hat{m}_{00} &\simeq \left| \bra{\psi}\ket{00} \right|^2 &= a^2 \\ 
	\hat{m}_{01} &\simeq \left| \bra{\psi}\ket{01} \right|^2 &= b^2 \\ 
	\hat{m}_{10} &\simeq \left| \bra{\psi}\ket{10} \right|^2 &= c^2 \\ 
	\hat{m}_{11} &\simeq \left| \bra{\psi}\ket{11} \right|^2 &= d^2  
\end{align}
$$

Next, using Hadamard gate on first qubit, we have:
$$
\begin{align}
	\hat{m}_{+0} &\simeq \left| \bra{\psi}\ket{+0} \right|^2 &= \left| \frac{a+c e^{i\theta_c}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{+1} &\simeq \left| \bra{\psi}\ket{+1} \right|^2 &= \left| \frac{b e^{i\theta_b}+d e^{i\theta_d}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{-0} &\simeq \left| \bra{\psi}\ket{-0} \right|^2 &= \left| \frac{a-c e^{i\theta_c}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{-1} &\simeq \left| \bra{\psi}\ket{-1} \right|^2 &= \left| \frac{b e^{i\theta_b}-d e^{i\theta_d}}{\sqrt{2}}\right|^2
\end{align}
$$
Then, we see:
$$
\begin{align}
	\cos\theta_{c} &= \frac{\hat{m}_{+0} - \hat{m}_{-0}}{2}\\
	\cos(\theta_{b} - \theta_{d}) &= \frac{\hat{m}_{+1} - \hat{m}_{-1}}{2}\\
\end{align}
$$
With Hadamard gate and S gate on first qubit, we see:
$$
\begin{align}
	\hat{m}_{i0} &\simeq \left| \bra{\psi}\ket{i0} \right|^2 &= \left| \frac{a+c e^{i\theta_c + \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{i1} &\simeq \left| \bra{\psi}\ket{i1} \right|^2 &= \left| \frac{b e^{i\theta_b}+d e^{i\theta_d + \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{-i0} &\simeq \left| \bra{\psi}\ket{-i0} \right|^2 &= \left| \frac{a-c e^{i\theta_c - \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{-i1} &\simeq \left| \bra{\psi}\ket{-i1} \right|^2 &= \left| \frac{b e^{i\theta_b}-d e^{i\theta_d - \frac{\pi}{2}}}{\sqrt{2}}\right|^2
\end{align}
$$
Then, 
$$
\begin{align}
	\sin\theta_{c} &= \frac{\hat{m}_{+0} - \hat{m}_{-0}}{2}\\
	\sin(\theta_{b} - \theta_{d}) &= \frac{\hat{m}_{+1} - \hat{m}_{-1}}{2}\\
\end{align}
$$
Also, using Hadamard gate on second qubit, we have:
$$
\begin{align}
	\hat{m}_{0+} &\simeq \left| \bra{\psi}\ket{0+} \right|^2 &= \left| \frac{a+b e^{i\theta_b}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{0-} &\simeq \left| \bra{\psi}\ket{0-} \right|^2 &= \left| \frac{a -b e^{i\theta_b}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{1+} &\simeq \left| \bra{\psi}\ket{1+} \right|^2 &= \left| \frac{c e^{i\theta_c} +d e^{i\theta_d}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{1-} &\simeq \left| \bra{\psi}\ket{1-} \right|^2 &= \left| \frac{c e^{i\theta_c}- d e^{i\theta_d}}{\sqrt{2}}\right|^2
\end{align}
$$
Then, we see:
$$
\begin{align}
	\cos\theta_{b} &= \frac{\hat{m}_{0+} - \hat{m}_{0-}}{2}\\
	\cos(\theta_{c} - \theta_{d}) &= \frac{\hat{m}_{1+} - \hat{m}_{1-}}{2}\\
\end{align}
$$
With Hadamard gate and S gate on second qubit, we see:
$$
\begin{align}
	\hat{m}_{0i} &\simeq \left| \bra{\psi}\ket{0i} \right|^2 &= \left| \frac{a+b e^{i\theta_b + \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{1i} &\simeq \left| \bra{\psi}\ket{1i} \right|^2 &= \left| \frac{a+b e^{i\theta_b + \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{0-i} &\simeq \left| \bra{\psi}\ket{0-i} \right|^2 &= \left| \frac{c e^{i\theta_c} +d e^{i\theta_d - \frac{\pi}{2}}}{\sqrt{2}}\right|^2 \\ 
	\hat{m}_{0-i} &\simeq \left| \bra{\psi}\ket{0-i} \right|^2 &= \left| \frac{c e^{i\theta_c}- d e^{i\theta_d - \frac{\pi}{2}}}{\sqrt{2}}\right|^2
\end{align}
$$
Then, 
$$
\begin{align}
	\sin\theta_{b} &= \frac{\hat{m}_{+0} - \hat{m}_{-0}}{2}\\
	\sin(\theta_{c} - \theta_{d}) &= \frac{\hat{m}_{+1} - \hat{m}_{-1}}{2}\\
\end{align}
$$
Now we can determine $\theta_{b}, \theta_{c}$ $\theta_{d}$.

#### Part III

### Problem 2

In [32]:
U_hidden = generate_clifford_circ(3, 3)

2 1 0
1 0 1
2 0 2


In [34]:
U_hidden.measure_all()